# Qwen3-4B: export a model for local CPU execution

**You are here → 05 — Qwen with ExecuTorch.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| **→ Optional CPU portability (current)** | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| Optional compatibility study | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

The earlier chapters called a model through a running server. Here we explore
another deployment path: package **Qwen3-4B** as a program an application can
load and execute locally with **ExecuTorch**. We use the same base checkpoint
revision as chapter 01, on an **Apple arm64 CPU**.

We'll follow the conversion and export steps, then inspect one generated SQL
answer and its timing. The exported model uses **XNNPACK**, a CPU execution backend,
with **INT4 linear weights and dynamic INT8 activations**. That is a different
precision recipe from AWQ W4A16, where activations remain FP16.

Choose **Run All** in the [notebook environment](../SETUP.md) to display the completed
conversion checks, native-run measurements and SQL result. This default needs no
model download or ExecuTorch installation. To perform a new export and model run,
use the stages in [the CPU export guide](../tutorial/EXECUTORCH_MAC.md).

The workload here is **one direct SQL question with a short schema**, rather than
the multi-step planning and repair workflow in chapters 01–04. We'll use it to
check portability, not to rank CPU and GPU performance.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.edge_notebook import recorded_qwen_edge
edge = recorded_qwen_edge(ROOT)
print(edge['label'])
display(JSON({k: edge[k] for k in ('source', 'source_sha256', 'model_name', 'model_revision',
    'model_sha256', 'tokenizer_sha256', 'backend', 'hardware', 'dtype', 'quantization',
    'max_context_length', 'threads', 'versions')}))
display(JSON({'matches_Dell_base_snapshot': edge['sql_smoke']['matches_dell_base_snapshot'],
              'base_snapshot_sha256': edge['sql_smoke']['base_snapshot_sha256']}))


recorded Mac CPU portability measurement — Qwen3-4B, November 9, 2026


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## 1. Turn the checkpoint into an executable model

The model weights alone are not enough for an application to execute. We need to
map them into ExecuTorch's model definition, verify that mapping and export the
operations for the selected backend:

**Qwen3-4B checkpoint → Qwen3 weight conversion → FP32 output checks → quantized
XNNPACK export → `.pte` program → native execution.**

A **`.pte` file** contains the exported program and weights for the ExecuTorch
runtime. Before quantizing, we compare the converted model's output logits—the
scores used to choose the next token—with the Hugging Face FP32 model on two
input sequences. The maximum absolute difference was **0.000044823**, within the
configured tolerance. This checks the mapping; SQL quality after quantization
still needs its own test.

The next cell shows the pinned model identity and export stages. The base snapshot
matches the Qwen FP16 baseline in chapter 01, including weights, tokenizer and config.

The first native load reported a missing quantized embedding operator. Registering
`executorch.kernels.quantized` made that operator available, and the same `.pte`
then ran. This illustrates why export and runtime loading are separate checks.
The [failure and fix](../experiment-tracker/tutorial/edge/qwen3-4b-kernel-registration-diagnosis.json)
are retained for troubleshooting.


In [2]:
display(JSON(edge['checkpoint_validation']))
display(JSON(edge['artifact_lineage']))


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## 2. Inspect the generated SQL and measure each stage

The next cell separates **model loading**, **time to first generated token** and
**complete inference**, which includes processing the prompt and generating the
answer. These timings come from ExecuTorch's native runner. Whole-process time
also includes Python imports and file verification, so it answers a different
startup question.

For memory, the table reports sampled **process RSS**: memory held by this CPU
process, including Python and runtime libraries. It is not GPU memory. The model
footprint counts the `.pte` and tokenizer files, rather than all installed software.

Read the raw generated SQL, then the separate result check. The query was executed
against the synthetic database and its rows matched the reference for **eval_001**.
Only the terminal `<|im_end|>` marker was removed before execution; SQL scoring is
outside generation timing. There was no planning or repair loop in this request.


In [3]:
rows = [
    ('Native model load', f"{edge['startup_seconds']:.3f} s"),
    ('Native first generated token', f"{edge['native_ttft_seconds']:.3f} s"),
    ('Native inference, including prompt evaluation', f"{edge['single_request_seconds']:.3f} s"),
    ('Whole process, including imports and artifact verification', f"{edge['wall_seconds']:.3f} s"),
    ('PTE + tokenizer footprint', f"{edge['footprint_bytes'] / 2**20:.2f} MiB"),
    ('Sampled peak process RSS', f"{edge['memory']['peak_bytes'] / 2**20:.2f} MiB"),
    ('Native prompt/generated tokens', f"{edge['generation']['num_prompt_tokens']} / {edge['generation']['num_generated_tokens']}"),
    ('Reviewed operator fallback', 'unavailable'),
]
display(Markdown('| Metric | Recorded Qwen CPU value |\n|---|---|\n' +
                 '\n'.join(f'| {name} | {value} |' for name, value in rows)))
display(JSON({'raw_generated_text': edge['generation']['text'],
              'sql_smoke': edge['sql_smoke'], 'limitations': edge['limitations']}))


| Metric | Recorded Qwen CPU value |
|---|---|
| Native model load | 1.688 s |
| Native first generated token | 0.195 s |
| Native inference, including prompt evaluation | 0.876 s |
| Whole process, including imports and artifact verification | 8.869 s |
| PTE + tokenizer footprint | 2365.90 MiB |
| Sampled peak process RSS | 5028.30 MiB |
| Native prompt/generated tokens | 52 / 29 |
| Reviewed operator fallback | unavailable |

<IPython.core.display.JSON object>

## 3. What can we take from this CPU run?

Qwen3-4B loaded and generated SQL through native ExecuTorch. Model load took
**1.688 s**, the first token arrived in **0.195 s**, and inference completed in
**0.876 s**, for **52 prompt tokens and 29 generated tokens**. The generated SQL
executed and matched the reference result.

That gives us a working starting point for embedding this model in a local
application. Before choosing it for a larger workload, test the context length,
question set and quantized outputs your application needs.

These are measurements of **one request**, after file checks had warmed the
filesystem cache. They don't establish stable tail latency or a CPU/GPU speed
advantage. The unquantized conversion checks also don't establish general SQL
accuracy after quantization, and runtime operator fallback hasn't been reviewed.

### An earlier small-model example

The [SmolLM2-135M run](../experiment-tracker/tutorial/edge/20261029T161510_034051Z.json)
and [its notebook](history/edge-smollm2-before-qwen.ipynb) remain available as a
separate example. They used FP32 and a different prompt. This chapter's result
comes from Qwen itself.

Continue to [Nemotron's compatibility check](06_nemotron_executorch.ipynb) or
[the deployment decision guide](../tutorial/RESULTS.md#what-should-i-choose).
